<a href="https://colab.research.google.com/github/Eleonwra/elcardiocc-baseline-ner/blob/main/model_training/mbert_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Importing Dataset

In [ ]:
%pip install datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 507.1/507.1 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.4 MB/s eta 0:00:00


In [ ]:
import pickle
import datasets

###Stanza

In [ ]:
with open('final_dataset_sentences_stanza.pickle', 'rb') as data:
    final_dataset_sentences = pickle.load(data)

###MedSpacy

In [ ]:
with open('final_dataset_sentences_medspacy.pickle', 'rb') as data:
    final_dataset_sentences = pickle.load(data)

##Checking lengths

In [ ]:
from transformers import BertTokenizerFast
tokenizer = BertTokenizerFast.from_pretrained("bert-base-multilingual-cased")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/29.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

In [ ]:
subset = ['train', 'validation', 'test']
tokenized_data = {'train': [], 'validation': [], 'test': []}
max_lengths = {name: 0 for name in subset}
sent_lengths = {'train': [], 'validation': [], 'test': []}
for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        texts =  ["".join(t) for t in section_data[i]["sentences_tokens"]]
        tokenized_document = tokenizer(texts)
        for j in range(len(tokenized_document['input_ids'])):
            tokenized_data[section_name].append(tokenized_document)
            sent_lengths[section_name].append(len(tokenized_document['input_ids'][j]))
            max_lengths[section_name] = max(max_lengths[section_name], len(tokenized_document['input_ids'][j]))
print("Max Length of sentences:", max_lengths)

Max Length of sentences: {'train': 458, 'validation': 303, 'test': 367}


In [ ]:
max_lengths_extra_info = {'train': 0, 'validation': 0, 'test': 0}
special_tokens = ["SEP", "CLS", "PAD"]
extra_len = {'train': [], 'validation': [], 'test': []}

for split in ['train', 'validation', 'test']:
    extra_info = []
    tokenized_extra_info = []
    labels_extra_info = []

    for i in range(len(final_dataset_sentences[split])):
        current_extra_info = final_dataset_sentences[split][i]['extra_info']
        current_extra_info = '[SEP]'.join(current_extra_info)
        current_tokenized_info = tokenizer(current_extra_info)
        current_input_ids = current_tokenized_info['input_ids']
        current_labels = [0 if any(token in tokenizer.convert_ids_to_tokens(current_input_ids[j]) for token in special_tokens) else 2 for j in range(len(current_input_ids))]

        extra_info.append(current_extra_info)
        tokenized_extra_info.append(tokenizer(current_extra_info))
        labels_extra_info.append(current_labels)

    for i in range(len(tokenized_extra_info)):
        tokenized_extra_info[i] = {
            'input_ids': tokenized_extra_info[i]['input_ids'],
            'token_type_ids': tokenized_extra_info[i]['token_type_ids'],
            'attention_mask': tokenized_extra_info[i]['attention_mask'],
            'labels': labels_extra_info[i]
        }
        for j in range(len(final_dataset_sentences[split][i]['sentences_tokens'])):
            extra_len[split].append(len(tokenized_extra_info[i]['input_ids']))

    max_lengths_extra_info[split] = max(len(doc['input_ids']) for doc in tokenized_extra_info)

print("Max Length for Extra Info:", max_lengths_extra_info)

Max Length for Extra Info: {'train': 195, 'validation': 135, 'test': 148}


In [ ]:
def count_elements_greater_than_threshold(values, threshold):
    return sum(1 for value in values if value > threshold)

for split in ['train', 'validation', 'test']:
    threshold = 512 - max_lengths_extra_info[split]
    count_greater_than_threshold = count_elements_greater_than_threshold(sent_lengths[split], threshold)
    print(f"Number of elements greater than {threshold} in {split}: {count_greater_than_threshold}")

Number of elements greater than 317 in train: 8
Number of elements greater than 377 in validation: 0
Number of elements greater than 364 in test: 1


##Tokenization

### Sentences with extra labels

In [ ]:
def tokenize_and_align_labels(data_section, max_lengths_extra_info):
    special_tokens = ["SEP", "CLS", "PAD"]
    texts = ["".join(t) for t in data_section["sentences_tokens"]]
    padded_texts = []
    for i in range(len(texts)):
      if 512 - max_lengths_extra_info > len(texts[i]):
        max_len = 512 - max_lengths_extra_info
        pad_len = max_len - len(texts[i])
        padded_texts.append((texts[i] + ' [PAD] ' * pad_len))
      else:
        padded_texts.append(texts[i][:max_len])

    tokenized_texts = tokenizer(padded_texts)
    if  len(data_section['extra_info']) != 0:
        extra_texts = data_section['extra_info']
        extra_texts = ' [SEP] '.join(extra_texts)
        tokenized_extra_texts = tokenizer(extra_texts)
        extra_input_ids = tokenized_extra_texts['input_ids']
        extra_labels = [0 if any(token in tokenizer.convert_ids_to_tokens(extra_input_ids[j]) for token in special_tokens) else 1 for j in range(len(extra_input_ids))]
        con_texts = [padded_texts[i] + ' [SEP] ' + extra_texts for i in range(len(padded_texts))]
        tokenized_inputs = tokenizer(con_texts, padding= 'max_length')
    else:
        tokenized_inputs = tokenizer(padded_texts)
        extra_labels = []
    attention_masks = tokenized_inputs['attention_mask']
    labels = []
    for row_idx, label_old in enumerate(data_section["sentences_ner_tags"]):
        label_new = [[] for t in tokenized_inputs.tokens(batch_index=row_idx)]
        for char_idx in range(len(data_section["sentences_tokens"][row_idx])):
            token_idx = tokenized_inputs.char_to_token(row_idx, char_idx)
            if token_idx is not None:
                label_new[token_idx].append(data_section["sentences_ner_tags"][row_idx][char_idx])
        label_new = list(map(lambda i : max(i, default=0), label_new))
        labels.append(label_new)
    tokenized_inputs["labels"] = [
    labels[i][:len(tokenized_texts[i]) - 1] + extra_labels + [0] * (512 + 1 - len(tokenized_texts[i])-len(extra_labels))for i in range(len(labels))]
    attention_masks = [[0 if token == tokenizer.pad_token_id else 1 for token in tokens] for tokens in tokenized_inputs["input_ids"]]
    tokenized_inputs["attention_mask"] = attention_masks
    return tokenized_inputs

In [ ]:
section_name = ['train', 'validation', 'test']
tokenized_data = {'train': [], 'validation': [], 'test': []}

for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        tokenized_document = tokenize_and_align_labels(section_data[i], max_lengths_extra_info[section_name])
        tokenized_data[section_name].append(tokenized_document)

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
result_df = pd.DataFrame(
    [tokenizer.convert_ids_to_tokens(tokenized_data['train'][1]["input_ids"][2]), tokenized_data['train'][1]["labels"][2],  tokenized_data['train'][1]["attention_mask"][2]],
    index=["tokens", "ner_tags", 'attention_Mask'])
print(result_df)

                  0   1    2    3    4    5    6    7    8      9   10   11   \
tokens          [CLS]   Ι  ##Σ  ##Τ  ##Ο  ##Ρ  ##Ι  ##Κ  ##Ο  [UNK]   Α  ##Ν   
ner_tags            0   0    0    0    0    0    0    0    0      0   0    0   
attention_Mask      1   1    1    1    1    1    1    1    1      1   1    1   

                12   13   14   15   16   17   18   19   20   21   22  23   \
tokens          ##Τ  ##Ι  ##Κ  ##Ε  ##Ι  ##Μ  ##Ε  ##Ν  ##Ι  ##Κ  ##Η   Ε   
ner_tags          0    0    0    0    0    0    0    0    0    0    0   0   
attention_Mask    1    1    1    1    1    1    1    1    1    1    1   1   

                24   25   26   27   28   29   30   31  32  33   34    35   \
tokens          ##Ξ  ##Ε  ##Τ  ##Ε  ##Τ  ##Α  ##Σ  ##Η   Η   α  ##σ  ##θε   
ner_tags          0    0    0    0    0    0    0    0   0   0    0     0   
attention_Mask    1    1    1    1    1    1    1    1   1   1    1     1   

                  36  37    38    39   40    41  42   43   44

###Only sentences

In [ ]:
def tokenize_and_align_labels(data_section, max_length):
    texts = ["".join(t) for t in data_section["sentences_tokens"]]
    tokenized_inputs = tokenizer(texts, padding = 'max_length')
    labels = []
    for row_idx, label_old in enumerate(data_section["sentences_ner_tags"]):
        label_new = [[] for t in tokenized_inputs.tokens(batch_index=row_idx)]
        for char_idx in range(len(data_section["sentences_tokens"][row_idx])):
            token_idx = tokenized_inputs.char_to_token(row_idx, char_idx)
            if token_idx is not None:
                label_new[token_idx].append(data_section["sentences_ner_tags"][row_idx][char_idx])
        label_new = list(map(lambda i : max(i, default=0), label_new))
        labels.append(label_new)
    tokenized_inputs["labels"] = labels + [0] * (512 - len(labels))
    return tokenized_inputs

In [ ]:
section_name = ['train', 'validation', 'test']
tokenized_data = {'train': [], 'validation': [], 'test': []}

for section_name, section_data in final_dataset_sentences.items():
    for i in range(len(section_data)):
        tokenized_document = tokenize_and_align_labels(section_data[i], max_lengths[section_name])
        tokenized_data[section_name].append(tokenized_document)

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
result_df = pd.DataFrame(
    [tokenizer.convert_ids_to_tokens(tokenized_data['train'][1]["input_ids"][2]), tokenized_data['train'][1]["labels"][2],  tokenized_data['train'][1]["attention_mask"][2]],
    index=["tokens", "ner_tags", 'attention_Mask'])
print(result_df)

                  0   1    2    3    4    5    6    7    8      9   10   11   \
tokens          [CLS]   Ι  ##Σ  ##Τ  ##Ο  ##Ρ  ##Ι  ##Κ  ##Ο  [UNK]   Α  ##Ν   
ner_tags            0   0    0    0    0    0    0    0    0      0   0    0   
attention_Mask      1   1    1    1    1    1    1    1    1      1   1    1   

                12   13   14   15   16   17   18   19   20   21   22  23   \
tokens          ##Τ  ##Ι  ##Κ  ##Ε  ##Ι  ##Μ  ##Ε  ##Ν  ##Ι  ##Κ  ##Η   Ε   
ner_tags          0    0    0    0    0    0    0    0    0    0    0   0   
attention_Mask    1    1    1    1    1    1    1    1    1    1    1   1   

                24   25   26   27   28   29   30   31  32  33   34    35   \
tokens          ##Ξ  ##Ε  ##Τ  ##Ε  ##Τ  ##Α  ##Σ  ##Η   Η   α  ##σ  ##θε   
ner_tags          0    0    0    0    0    0    0    0   0   0    0     0   
attention_Mask    1    1    1    1    1    1    1    1   1   1    1     1   

                  36  37    38    39   40    41  42   43   44

##Save

In [ ]:
with open('mBERT_tokenized_documents_stanza.pkl', 'wb') as f:
    pickle.dump(tokenized_data, f)